In [1]:
import numpy as np
import os
import matplotlib.pyplot as plt
import cmocean
import matplotlib as mpl
import scipy.io as sio
import xarray as xr
import gc
from cftime import date2num

def MOM_from_LES(
    theta,     # (ntime, nz) LES temperature [°C], on zl
    tw,        # (ntime, nz+1) LES <w'θ'> in K·m/s, on zi
    tau13l,    # (ntime,)   LES τ13l [Pa]
    tau23l,    # (ntime,)   LES τ23l [Pa]
    zl,        # (nz,)      temperature depth grid [m], positive downward
    zi,        # (nz+1,)    interface depth grid [m], for tw
    time,
    HbL_l=None,
    rho0=1026.95, alpha=-0.2,
    g=9.81, nstar=0.06
):
    """
    Returns
      Hbl    : (ntime,) mixed‐layer depth by max |dθ/dz| on zl
      Me     : (ntime,) entrainment PE conversion ∫ w'b' dz on zi
      mstar  : (ntime,) mechanical coefficient from Eq.(13)
      ustar  : (ntime,) friction velocity
    """
    ntime, nz = theta.shape
    nz_zi = len(zi)

    # --- 1. Compute ∂T/∂z on zi (to match tw) ---
    dTdz = np.empty((ntime, nz_zi)) * np.nan
    dtheta = np.gradient(theta, zl, axis=1)  # dT/dz on zl (nz)

    # Interpolate ∂T/∂z from zl to zi
    dTdz[:,1:-1] = 0.5 * (dtheta[:, :-1] + dtheta[:, 1:])
    dTdz[:, 0]   = dtheta[:, 0]     # top boundary
    dTdz[:, -1]  = dtheta[:, -1]    # bottom boundary

    # --- 2. Compute buoyancy flux on zi ---
    bflux = -(g / rho0) * alpha * tw  # shape (ntime, nz+1)

    # --- 3. MLD from θ gradient on zl ---
    idx  = np.argmax(np.abs(dtheta), axis=1)   # max |∂θ/∂z|
    if HbL_l is None:
        Hbl  = zl[idx]                             # depth of MLD
    else:
        Hbl=HbL_l

    # --- 4. Loop over time steps to integrate Me, conv, compute mstar ---
    Me    = np.zeros(ntime)
    conv  = np.zeros(ntime)
    mstar = np.zeros(ntime)

    tau_mag = np.hypot(tau13l, tau23l)
    ustar   = np.sqrt(tau_mag / rho0)

    for t in range(ntime):
        H = Hbl[t]
        if np.isnan(H) or H <= zi[0]:
            mstar[t] = np.nan
            continue

        ml = zi <= H
        zz = zi[ml]
        bf = bflux[t, ml]

        # Me: negative buoyancy flux only (entrainment)
        neg = bf < 0
        Me[t] = np.trapezoid(bf[neg], zz[neg]) if np.any(neg) else 0.0

        # convective buoyancy flux
        pos = bf > 0
        conv[t] = np.trapezoid(bf[pos], zz[pos]) if np.any(pos) else 0.0

        # Eq. 13 inversion
        if ustar[t] > 0:
            mstar[t] = (-Me[t] - nstar * conv[t]) / (ustar[t]**3)
        else:
            mstar[t] = np.nan

    return Hbl, Me, mstar, ustar

y_km_list     = [50,30,10,0,-20,-40,-60,-200]
y_folder_list = ['50','30','10','0','S20','S40','S60','S200']

rho0=1026.95 # reference density [kg/m³]
nstar=0.06  # convective coefficient
alpha=-0.2   # thermal expansion coeff [kg·m⁻³ °C⁻¹]
g=9.81        # gravity [m/s²] 
dt_f=300
dt = dt_f/(24*3600)  # For example, 60 seconds one minutes, shorter time in MOM, 5 minutes
##attention mstar dt time step >= MOM time stepDT
save_dir = "/archive/Qian.Xiao/Qian.Xiao/MOM-examples/SCM_ePBL_shearimposedmstarShear/"
os.makedirs(save_dir,exist_ok=True)
   
for idx, y_km in enumerate(y_km_list):
    y_tag    = y_folder_list[idx]
    base_dir = f'/archive/Qian.Xiao/Qian.Xiao/MOM-examples/SCM_idealized_hurricane_shear/10mps_300sResults/{y_tag}_results'
    MOM_sf     = xr.open_dataset(os.path.join(base_dir, 'surffluxes.nc')).isel(xh=0, yh=0, xq=0, yq=0)
    MOM_prog   = xr.open_dataset(os.path.join(base_dir, 'prog.nc')).isel(xh=0, yh=0, xq=0, yq=0)
    MOM_visc   = xr.open_dataset(os.path.join(base_dir, 'visc.nc')).isel(xh=0, yh=0, xq=0, yq=0)

    Mtime=MOM_sf.Time
    ref = Mtime.values[0]
    time_days = date2num(Mtime.values, units=f'days since {ref.year}-{ref.month:02d}-{ref.day:02d}', calendar='julian')
    MOMtime=time_days

    # ==== Compute MOM6 mstar ====
    Hbl_mom, Me_mom, mstar_mom, ustar_mom = MOM_from_LES(
        MOM_prog.temp,
        -MOM_visc.Tflx_dia_diff,
        MOM_sf.taux,
        MOM_sf.tauy,
        MOM_prog.zl,
        MOM_visc.zi,
        MOMtime,
        HbL_l=None,
        rho0=rho0,
        alpha=alpha,
        g=g,
        nstar=nstar,
    )
    
    time_uniform = np.arange(0, MOMtime[-1] + dt, dt)
    mstar_interp_LES_noLT = np.interp(time_uniform, MOMtime, mstar_mom)
    
    # ==== Save MOM6 time and mstar ====
    save_path = os.path.join(save_dir, f"LES_noLT_results_10mps/dt_{dt_f}/time_{y_tag}_LES_noLT.txt")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)  # Ensure directory exists
    np.savetxt(save_path, time_uniform, fmt="%.6f")
    
    save_path = os.path.join(save_dir, f"LES_noLT_results_10mps/dt_{dt_f}/mstar_{y_tag}_LES_noLT.txt")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)  # Ensure directory exists
    np.savetxt(save_path, mstar_interp_LES_noLT, fmt="%.6f")

    del MOM_sf, MOM_prog, MOM_visc
    gc.collect()
    


 




    

Matplotlib is building the font cache; this may take a moment.
/vftmp/Qian.Xiao/pid3070850/ipykernel_3090613/3921904180.py:102: SerializationWarning: Unable to decode time axis into full numpy.datetime64 objects, continuing using cftime.datetime objects instead, reason: dates prior reform date (1582-10-15). To silence this warning specify 'use_cftime=True'.
  MOM_sf     = xr.open_dataset(os.path.join(base_dir, 'surffluxes.nc')).isel(xh=0, yh=0, xq=0, yq=0)
/vftmp/Qian.Xiao/pid3070850/ipykernel_3090613/3921904180.py:102: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  MOM_sf     = xr.open_dataset(os.path.join(base_dir, 'surffluxes.nc')).isel(xh=0, yh=0, xq=0, yq=0)
/vftmp/Qian.Xiao/pid3070850/ipykernel_3090613/3921904180.py:104: SerializationWarning: Unable to decode time axis into full numpy.datetime64 objects, continuing using cftime.d